# TACTIC-FP Segment Slicer

This notebook bridges the **master trajectory** (one `.npz` per match from YOLOv11 + Deep-EIoU) to **per-segment `.npz` files** required by the TACTIC-FP training pipeline.

**What it does:**
1. Loads the full-match trajectory `[T_total, 23, 4]`
2. Reads the annotation JSON with segment boundaries
3. Crops each segment into its own `.npz` file
4. Validates shapes, padding masks, and path uniqueness
5. Writes a corrected JSON with validated metadata

> **Run this notebook after:** annotating the match and generating the master `.npz` from the tracking pipeline.
> **Run this notebook before:** model training.

In [ ]:
import numpy as np
import json
from pathlib import Path
from typing import List, Tuple, Dict, Any

print("✅ Imports ready")

## Step 1 — Configuration

Set the paths to your **master `.npz`** (from tracking) and **annotation JSON** (from the annotator tool).

Also set the operating FPS (`10` for TACTIC-FP) and the max frame window (`150` = 15 s).

In [ ]:
# ─── USER CONFIGURATION ───
MASTER_NPZ_PATH = "data/masters/match_001_master.npz"   # YOLOv11 + Deep-EIoU output
ANNOTATION_JSON_PATH = "data/exports/TACTIC_FP_Annotated_match_001.json"
OUTPUT_ROOT = "data/trajectories"                        # Where segment .npz files will be saved
CORRECTED_JSON_PATH = "data/exports/TACTIC_FP_Annotated_match_001_sliced.json"

# ─── MODEL CONSTANTS ───
MODEL_FPS = 10.0          # TACTIC-FP operates at 10 fps
MAX_FRAMES = 150          # 15 seconds @ 10 fps
MIN_DURATION_MS = 2000    # 2 seconds minimum
MAX_DURATION_MS = 15000   # 15 seconds maximum

print(f"Master NPZ:  {MASTER_NPZ_PATH}")
print(f"Annotation:  {ANNOTATION_JSON_PATH}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Model FPS:   {MODEL_FPS} | Max frames: {MAX_FRAMES}")

## Step 2 — Load Master Trajectory

The master `.npz` contains the full-match tracking data. Shape should be `[T_total, 23, 4]`.

We handle common key names (`trajectory`, `data`, or the first available key).

In [ ]:
def load_master_trajectory(npz_path: str) -> np.ndarray:
    """Load the master trajectory and return the [T_total, 23, 4] array."""
    master = np.load(npz_path)
    
    if 'trajectory' in master:
        arr = master['trajectory']
    elif 'data' in master:
        arr = master['data']
    else:
        first_key = next(iter(master.keys()))
        arr = master[first_key]
        print(f"⚠️  Using first key '{first_key}' from master NPZ")
    
    print(f"✅ Master loaded: shape = {arr.shape}, dtype = {arr.dtype}")
    
    # Validate dimensions
    if len(arr.shape) != 3:
        raise ValueError(f"Expected 3D array, got {len(arr.shape)}D: {arr.shape}")
    if arr.shape[1] != 23 or arr.shape[2] != 4:
        raise ValueError(f"Expected (*, 23, 4), got {arr.shape}")
    
    return arr

master_array = load_master_trajectory(MASTER_NPZ_PATH)
T_total, n_agents, n_features = master_array.shape
print(f"Total frames: {T_total} | Agents: {n_agents} | Features: {n_features}")

## Step 3 — Load Annotation JSON

Read the annotator output. We expect the structure:
```
halves -> [ segments -> { start_ms, end_ms, reconstruction -> { npz_path, tensor_shape, ... } } ]
```

In [ ]:
def load_annotation_json(json_path: str) -> Dict[str, Any]:
    """Load and return the annotation JSON."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # Count segments
    seg_count = sum(len(h.get('segments', [])) for h in data.get('halves', []))
    print(f"✅ JSON loaded: {seg_count} segments across {len(data.get('halves', []))} half(es)")
    return data

annotation_data = load_annotation_json(ANNOTATION_JSON_PATH)
print(f"Match ID: {annotation_data.get('match_id', 'N/A')}")

## Step 4 — Slicer Function

Core logic:
1. For each segment, compute frame indices from `start_ms` / `end_ms` at `MODEL_FPS`.
2. Slice the master array.
3. Validate against the JSON's expected `tensor_shape`.
4. Save the segment `.npz`.
5. Update the JSON with the validated `tensor_shape` and `padding_mask`.

**Excluded segments** (`model_split == 'excluded'`) are skipped — no `.npz` is created for them.

In [ ]:
def slice_segments(
    master_array: np.ndarray,
    annotation_data: Dict[str, Any],
    output_root: str,
    fps: float = 10.0,
    max_frames: int = 150,
    min_duration_ms: int = 2000,
    max_duration_ms: int = 15000,
) -> Tuple[List[str], List[str], Dict[str, Any]]:
    """
    Slice master trajectory into per-segment .npz files.
    
    Returns:
        success_paths: list of created .npz file paths
        errors:      list of error messages
        corrected_json: annotation data with updated reconstruction metadata
    """
    T_total = master_array.shape[0]
    success_paths = []
    errors = []
    
    # Deep copy JSON so we can mutate it safely
    corrected_json = json.loads(json.dumps(annotation_data))
    
    # Track used npz paths to detect duplicates
    used_paths = set()
    
    for half_idx, half in enumerate(corrected_json.get('halves', [])):
        for seg_idx, seg in enumerate(half.get('segments', [])):
            seg_id = seg.get('segment_id', f'half{half_idx}_seg{seg_idx}')
            
            # ── Skip excluded segments ──
            if seg.get('model_split') == 'excluded':
                print(f"  ⏭️  {seg_id}: excluded (DeadBall/ContestedPlay)")
                continue
            
            # ── Read timing ──
            start_ms = seg.get('start_ms', 0)
            end_ms = seg.get('end_ms', 0)
            duration_ms = end_ms - start_ms
            
            # ── Duration gates ──
            if duration_ms <= 0:
                errors.append(f"{seg_id}: duration <= 0 ({duration_ms}ms)")
                continue
            if duration_ms < min_duration_ms:
                errors.append(f"{seg_id}: duration {duration_ms}ms < {min_duration_ms}ms minimum")
                continue
            if duration_ms > max_duration_ms:
                errors.append(f"{seg_id}: duration {duration_ms}ms > {max_duration_ms}ms maximum — split required")
                continue
            
            # ── Compute frame indices ──
            start_frame = int(round(start_ms / 1000.0 * fps))
            end_frame = int(round(end_ms / 1000.0 * fps))
            actual_frames = end_frame - start_frame
            
            if actual_frames <= 0:
                errors.append(f"{seg_id}: computed frames <= 0")
                continue
            
            if actual_frames > max_frames:
                errors.append(f"{seg_id}: {actual_frames} frames exceeds max {max_frames}")
                end_frame = start_frame + max_frames
                actual_frames = max_frames
            
            # ── Bounds check against master ──
            if end_frame > T_total:
                errors.append(f"{seg_id}: frame range [{start_frame}:{end_frame}] exceeds master [0:{T_total}]")
                continue
            
            # ── Slice ──
            segment_array = master_array[start_frame:end_frame].copy()
            
            # ── Validate against JSON expectation ──
            expected_shape = seg.get('reconstruction', {}).get('tensor_shape')
            if expected_shape and list(segment_array.shape) != expected_shape:
                errors.append(
                    f"{seg_id}: shape mismatch — JSON expects {expected_shape}, "
                    f"got {list(segment_array.shape)}. Trusting actual slice."
                )
            
            # ── Resolve output path ──
            npz_path = seg.get('reconstruction', {}).get('npz_path', '')
            if not npz_path:
                npz_path = f"data/trajectories/{annotation_data.get('match_id', 'match')}/{seg_id}.npz"
                seg['reconstruction']['npz_path'] = npz_path
            
            # Duplicate detection
            if npz_path in used_paths:
                # Auto-increment suffix
                stem = Path(npz_path).stem
                suffix = 1
                parent = str(Path(npz_path).parent)
                while f"{parent}/{stem}_v{suffix}.npz" in used_paths:
                    suffix += 1
                npz_path = f"{parent}/{stem}_v{suffix}.npz"
                seg['reconstruction']['npz_path'] = npz_path
                errors.append(f"{seg_id}: duplicate path resolved to {npz_path}")
            used_paths.add(npz_path)
            
            # ── Save ──
            out_path = Path(output_root) / npz_path
            out_path.parent.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(out_path, trajectory=segment_array)
            success_paths.append(str(out_path))
            
            # ── Update JSON metadata ──
            if 'reconstruction' not in seg:
                seg['reconstruction'] = {}
            seg['reconstruction']['tensor_shape'] = list(segment_array.shape)
            seg['reconstruction']['tensor_fps'] = fps
            seg['reconstruction']['quality_pass'] = True
            seg['reconstruction']['tracked_players'] = 22
            seg['reconstruction']['tracked_ball'] = True
            seg['reconstruction']['tracking_confidence_mean'] = seg.get('reconstruction', {}).get('tracking_confidence_mean', 0.85)
            
            # Padding mask: 1s for actual frames, 0s for rest up to max_frames
            padding_mask = [1] * actual_frames + [0] * (max_frames - actual_frames)
            padding_mask = padding_mask[:max_frames]
            seg['reconstruction']['padding_mask'] = padding_mask
            
            print(f"  ✅ {seg_id}: {actual_frames} frames -> {npz_path}")
    
    return success_paths, errors, corrected_json

print("✅ Slicer function defined")

## Step 5 — Run the Slicer

Execute the slice. This may take a few seconds depending on the number of segments.

In [ ]:
success_paths, errors, corrected_json = slice_segments(
    master_array=master_array,
    annotation_data=annotation_data,
    output_root=OUTPUT_ROOT,
    fps=MODEL_FPS,
    max_frames=MAX_FRAMES,
    min_duration_ms=MIN_DURATION_MS,
    max_duration_ms=MAX_DURATION_MS,
)

print(f"\n{'='*50}")
print(f"SLICE COMPLETE")
print(f"{'='*50}")
print(f"Created:   {len(success_paths)} segment .npz files")
print(f"Errors:    {len(errors)}")
if errors:
    print("\nError details:")
    for e in errors:
        print(f"  ❌ {e}")

## Step 6 — Validate Outputs

We verify:
1. Every `.npz` file exists and is readable.
2. Its shape matches the corrected JSON.
3. The padding mask has exactly `T` ones.
4. No excluded segments leaked into the success list.
5. No duplicate paths remain.

In [ ]:
def validate_outputs(corrected_json: Dict[str, Any], success_paths: List[str], max_frames: int = 150) -> Tuple[List[str], List[str]]:
    """Validate all created .npz files against the corrected JSON."""
    validation_ok = []
    validation_err = []
    
    # Build map of segment_id -> reconstruction
    seg_map = {}
    for half in corrected_json.get('halves', []):
        for seg in half.get('segments', []):
            seg_id = seg.get('segment_id')
            seg_map[seg_id] = seg
    
    # Check every success path
    for npz_file in success_paths:
        p = Path(npz_file)
        if not p.exists():
            validation_err.append(f"MISSING: {npz_file}")
            continue
        
        try:
            data = np.load(npz_file)
            traj = data['trajectory']
        except Exception as e:
            validation_err.append(f"CORRUPT: {npz_file} — {e}")
            continue
        
        # Find matching segment by npz_path
        rel_path = str(Path(npz_file).relative_to(OUTPUT_ROOT))
        matched_seg = None
        for seg_id, seg in seg_map.items():
            if seg.get('reconstruction', {}).get('npz_path') == rel_path:
                matched_seg = seg
                break
        
        if matched_seg is None:
            validation_err.append(f"ORPHAN: {rel_path} has no JSON entry")
            continue
        
        expected_shape = matched_seg['reconstruction']['tensor_shape']
        if list(traj.shape) != expected_shape:
            validation_err.append(
                f"SHAPE: {rel_path} — JSON {expected_shape} vs NPZ {list(traj.shape)}"
            )
            continue
        
        # Check padding mask
        mask = matched_seg['reconstruction']['padding_mask']
        ones = sum(mask)
        if ones != traj.shape[0]:
            validation_err.append(
                f"MASK: {rel_path} — {ones} ones vs {traj.shape[0]} frames"
            )
            continue
        
        if len(mask) != max_frames:
            validation_err.append(
                f"MASK_LEN: {rel_path} — mask length {len(mask)} != {max_frames}"
            )
            continue
        
        validation_ok.append(rel_path)
    
    # Check for duplicate paths in JSON
    all_paths = []
    for half in corrected_json.get('halves', []):
        for seg in half.get('segments', []):
            if seg.get('model_split') != 'excluded':
                p = seg.get('reconstruction', {}).get('npz_path')
                if p:
                    all_paths.append(p)
    
    from collections import Counter
    dupes = {p: c for p, c in Counter(all_paths).items() if c > 1}
    if dupes:
        for p, c in dupes.items():
            validation_err.append(f"DUPLICATE: {p} appears {c} times in JSON")
    
    return validation_ok, validation_err

print("✅ Validation function defined")

In [ ]:
validation_ok, validation_err = validate_outputs(corrected_json, success_paths, MAX_FRAMES)

print(f"\n{'='*50}")
print(f"VALIDATION RESULTS")
print(f"{'='*50}")
print(f"Passed:  {len(validation_ok)}")
print(f"Failed:  {len(validation_err)}")

if validation_err:
    print("\nFailed checks:")
    for e in validation_err:
        print(f"  ❌ {e}")
else:
    print("\n🎉 All validations passed. Data is training-ready.")

## Step 7 — Save Corrected JSON

Write the updated annotation manifest with validated `tensor_shape`, `tensor_fps`, and `padding_mask`.
This is the file you feed into the model dataloader.

In [ ]:
def save_corrected_json(corrected_json: Dict[str, Any], out_path: str):
    """Save the corrected annotation JSON."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(corrected_json, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Corrected JSON saved: {out_path}")
    
    # Print summary stats
    total = sum(len(h.get('segments', [])) for h in corrected_json.get('halves', []))
    excluded = sum(
        1 for h in corrected_json.get('halves', [])
        for s in h.get('segments', []) if s.get('model_split') == 'excluded'
    )
    trainable = total - excluded
    print(f"   Total segments:     {total}")
    print(f"   Excluded:           {excluded}")
    print(f"   Trainable:          {trainable}")

save_corrected_json(corrected_json, CORRECTED_JSON_PATH)

## Summary

You now have:
- One **master `.npz`** per match (from tracking)
- One **annotation JSON** per match (from the annotator tool)
- One **corrected JSON** with validated metadata
- Many **segment `.npz` files** (`[T_seg, 23, 4]`) ready for the dataloader

**Next step:** Feed the corrected JSON and segment `.npz` files into the TACTIC-FP dataloader.